# 08 — Avaliação final no conjunto de teste

Avalie os **quatro modelos × seeds 42, 123 e 456**, sem treinamento. O protocolo é fixo: os 12 checkpoints `*_best.pth` já escolhidos pela menor loss de **validação**, todas as **8.125 imagens de teste** do CSV original, 38 classes e nenhuma seleção posterior por desempenho de teste.

As métricas são descritivas da avaliação final: não escolher arquitetura, hiperparâmetros, época ou seed com base nelas. As médias e desvios padrão são calculados entre as três sementes (desvio **amostral, ddof=1**), sem misturar previsões ou escolher a melhor seed.

**A execução real está desativada por padrão**, inclusive ao usar “Executar tudo”. A preparação lê apenas CSVs, JSONs e checkpoints; não abre imagens. A extração e a avaliação exigem `EXECUTE_TEST_EVALUATION = True`.

No Colab, use a pasta do Drive com os resultados originais do notebook 07. Abrir este notebook pelo GitHub não sincroniza `src/` com o Drive. Atualize os scripts antes de começar e preserve os resultados existentes. Não execute os notebooks de split ou treinamento.

In [ ]:
from pathlib import Path
import sys
try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount('/content/drive')

PROJECT_ROOT = (Path('/content/drive/MyDrive/plant-disease-classification')
                if IN_COLAB else Path.cwd())
if not (PROJECT_ROOT / 'src').is_dir() and (PROJECT_ROOT.parent / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS_ROOT = PROJECT_ROOT / 'results'
IMAGE_ROOT = Path('/content/plantvillage_color') if IN_COLAB else PROJECT_ROOT / 'data/plantvillage_color'
ZIP_PATH = PROJECT_ROOT / 'data/data.zip'  # ajuste para o ZIP original
required = ('evaluate_test.py', 'compare_models.py', 'compare_repetitions.py',
            'run_repetition.py', 'experiment_utils.py', 'plantvillage_pytorch.py',
            'plantvillage_split.py', 'baseline_cnn.py', 'resnet50_model.py',
            'densenet201_model.py', 'efficientnet_b0_model.py')
missing = [name for name in required if not (PROJECT_ROOT / 'src' / name).is_file()]
if missing:
    raise FileNotFoundError(f'Sincronize a pasta src/ atualizada em {PROJECT_ROOT}: {missing}')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Projeto:', PROJECT_ROOT)
print('Resultados de treinamento:', RESULTS_ROOT)
print('Imagens locais:', IMAGE_ROOT)

In [ ]:
EXECUTE_TEST_EVALUATION = False  # True somente para iniciar a avaliação final deliberadamente
EXTRACT_TEST_IMAGES = False  # True se precisar extrair as imagens reservadas do ZIP
BATCH_SIZE = 32
NUM_WORKERS = 2  # use 0 para depuração; não altera o conjunto de imagens


## Preparação sem acesso às imagens

Exige as 12 execuções completas e valida os arquivos e metadados, hashes dos manifestos de repetições, seeds, isolamento de `leaf_id`, caminhos únicos, contagens 38.008/8.172/8.125 e classes. Carrega cada checkpoint na CPU com `weights_only=True` e `load_state_dict(strict=True)`; os modelos de transferência são construídos com `weights=None`, sem download de pesos.

Seed 42: `results/<modelo>/`. Seeds 123/456: `results/repetitions/seed_<seed>/<modelo>/`. Não basta ter a tabela consolidada do notebook 07: os arquivos individuais precisam estar presentes.

**Limite de proveniência:** checkpoints antigos não incorporam seed/modelo/hash dos splits. Sua associação é conferida pelos diretórios, metadados, manifestos, mapas de classes, época, métricas de validação e estrutura dos pesos. A correspondência histórica do CSV com a seed 42 depende de preservar o CSV original; os hashes registrados nas repetições são exigidos. Os avisos são preservados no manifesto final.

In [ ]:
from evaluate_test import prepare_test_evaluation
import pandas as pd

plan = prepare_test_evaluation(RESULTS_ROOT)
display(pd.DataFrame([{k: run[k] for k in ('model_name', 'seed', 'checkpoint')} for run in plan.runs]))
print('12 checkpoints verificados; nenhuma imagem aberta.')
for warning in plan.warnings:
    print('AVISO:', warning)


## Imagens de teste locais (opcional e protegida)

Se as imagens já estiverem extraídas, ajuste `IMAGE_ROOT` e mantenha `EXTRACT_TEST_IMAGES=False`. Se precisar extrair do ZIP, habilite as duas chaves. Esta célula extrai exclusivamente os caminhos `test` do CSV original, sem refazer splits ou abrir imagens. A integridade visual de todos os arquivos será verificada antes da inferência. Arquivos locais existentes são preservados.

In [ ]:
if EXECUTE_TEST_EVALUATION and EXTRACT_TEST_IMAGES:
    import shutil
    import zipfile
    from evaluate_test import assert_sources
    from plantvillage_pytorch import _raw_color_relative_parts
    assert_sources(plan.sources)
    selected = pd.read_csv(RESULTS_ROOT / 'plantvillage_metadata_split.csv')
    selected = selected.loc[selected['split'].eq('test'), 'zip_path']
    with zipfile.ZipFile(ZIP_PATH) as archive:
        for zip_path in selected:
            destination = IMAGE_ROOT.joinpath(*_raw_color_relative_parts(zip_path))
            if not destination.resolve().is_relative_to(IMAGE_ROOT.resolve()):
                raise ValueError('Destino fora de IMAGE_ROOT')
            member = archive.getinfo(zip_path)
            if destination.exists():
                if destination.stat().st_size != member.file_size:
                    raise ValueError(f'Arquivo existente com tamanho divergente: {destination}')
                continue
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, destination.open('xb') as target:
                shutil.copyfileobj(source, target)
    print('Extração dos caminhos de teste concluída.')
else:
    print('Extração desativada; nenhuma imagem acessada.')

## Execução deliberada

O pré-processamento reproduz o de validação dos scripts: RGB, resize 224×224, tensor e normalização ImageNet. A ResNet50 usa seu construtor de transforms existente; DenseNet201/EfficientNet-B0 verificam a compatibilidade com o pipeline compartilhado. Preserva-se a decisão de não aplicar a geometria de recorte do preset oficial da EfficientNet. Sem augmentation aleatório, shuffle, descarte de batches ou aquecimento adicional.

A avaliação usa `eval()` e `inference_mode()`, um modelo por vez, sem otimizador. O código valida novamente as fontes antes da execução e durante o processo; nenhuma imagem ausente/corrompida é ignorada. Uma falha deixa a pasta com status `failed`; não use saídas parciais como resultado final.

Cada execução reserva `results/test_evaluation/<data_UTC>_<id>/` com criação exclusiva. Uma segunda execução gera outra pasta. Não modifique a configuração em resposta às métricas do teste. Tempos são auxiliares, dependem do hardware/ambiente e não são incluídos nas médias das métricas de classificação.

In [ ]:
OUTPUT_DIR = None
if EXECUTE_TEST_EVALUATION:
    from evaluate_test import run_test_evaluation
    OUTPUT_DIR = run_test_evaluation(
        RESULTS_ROOT, IMAGE_ROOT, execute=True, plan=plan,
        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    )
    print('Avaliação final salva em:', OUTPUT_DIR)
else:
    print('Avaliação real desativada. Para executar, altere a chave na configuração e reexecute a partir dela.')

In [ ]:
if OUTPUT_DIR is not None:
    from evaluate_test import read_json
    manifest = read_json(OUTPUT_DIR / 'evaluation_manifest.json')
    if manifest['status'] != 'complete' or len(manifest['runs']) != 12:
        raise RuntimeError('Avaliação incompleta; não utilizar como resultado final.')
    display(pd.read_csv(OUTPUT_DIR / 'test_individual.csv'))
    display(pd.read_csv(OUTPUT_DIR / 'test_aggregate.csv'))
    print('Desvio padrão amostral entre sementes: ddof=1. Sem seleção pelo teste.')
else:
    print('Nenhum resultado real foi gerado nesta sessão.')

## Arquivos de saída

- `evaluation_manifest.json`: status global, protocolo, seeds, ambiente, fontes e resultados com SHA-256, avisos e execuções concluídas.
- `test_split.csv`, `class_to_idx.json`, `test_images_sha256.csv`: seleção exata e ordem das imagens avaliadas, mapa e hashes dos arquivos.
- `test_individual.csv` e `test_aggregate.csv`: sete métricas por execução e média/desvio amostral por modelo (três seeds).
- `test_by_class_individual.csv` e `test_by_class_aggregate.csv`: precisão, recall, F1 e suporte por classe; média/desvio entre seeds, sem somar o suporte três vezes.
- `<modelo>/seed_<seed>/`: métricas gerais/por classe, `predictions.csv` com caminho/rótulo/predição, matrizes de confusão absoluta e normalizada por classe real em CSV, matriz absoluta em PNG, tempo e pré-processamento em `inference.json`.

As linhas das matrizes são classes reais; as colunas são predições, na ordem de `class_to_idx`. As métricas ficam em [0, 1], usam todas as 38 classes e `zero_division=0`. Matrizes são individuais por seed; não são tratadas como 24.375 observações independentes.